In [52]:
import numpy as np
import tensorflow as tf

In [53]:
text = "hello how are you doing today have a good day take care stay safe good morning good night thank you for your support you are welcome see you soon hope everything works out fine keep going one step at a time learning new things is fun take breaks drink water stay consistent deep learning models learn patterns GRU remembers context efficiently LSTM remembers longer dependencies "

### Create character vocab

In [54]:
chars = sorted(set(text))
print(chars)

char2id = {c:i for i,c in enumerate(chars)}
print(char2id)

id2char = {i:c for i,c in enumerate(chars)}
print(id2char)

[' ', 'G', 'L', 'M', 'R', 'S', 'T', 'U', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'k', 'l', 'm', 'n', 'o', 'p', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y']
{' ': 0, 'G': 1, 'L': 2, 'M': 3, 'R': 4, 'S': 5, 'T': 6, 'U': 7, 'a': 8, 'b': 9, 'c': 10, 'd': 11, 'e': 12, 'f': 13, 'g': 14, 'h': 15, 'i': 16, 'k': 17, 'l': 18, 'm': 19, 'n': 20, 'o': 21, 'p': 22, 'r': 23, 's': 24, 't': 25, 'u': 26, 'v': 27, 'w': 28, 'x': 29, 'y': 30}
{0: ' ', 1: 'G', 2: 'L', 3: 'M', 4: 'R', 5: 'S', 6: 'T', 7: 'U', 8: 'a', 9: 'b', 10: 'c', 11: 'd', 12: 'e', 13: 'f', 14: 'g', 15: 'h', 16: 'i', 17: 'k', 18: 'l', 19: 'm', 20: 'n', 21: 'o', 22: 'p', 23: 'r', 24: 's', 25: 't', 26: 'u', 27: 'v', 28: 'w', 29: 'x', 30: 'y'}


In [55]:
vocab_size = len(chars)
print(vocab_size)

31


### Create training pairs

In [82]:
# X = 1st 3 characters, y = 4th character
X, y = [], []
for i in range(len(text)-3):
  X.append([char2id[c] for c in text[i:i+3]])
  y.append(char2id[text[i+3]])
  if 0<=i<=5:
    print("X :- ", [char2id[c] for c in text[i:i+3]], "and y :-", char2id[text[i+3]])

X :-  [15, 12, 18] and y :- 18
X :-  [12, 18, 18] and y :- 21
X :-  [18, 18, 21] and y :- 0
X :-  [18, 21, 0] and y :- 15
X :-  [21, 0, 15] and y :- 21
X :-  [0, 15, 21] and y :- 28


In [57]:
print(X)

[[15, 12, 18], [12, 18, 18], [18, 18, 21], [18, 21, 0], [21, 0, 15], [0, 15, 21], [15, 21, 28], [21, 28, 0], [28, 0, 8], [0, 8, 23], [8, 23, 12], [23, 12, 0], [12, 0, 30], [0, 30, 21], [30, 21, 26], [21, 26, 0], [26, 0, 11], [0, 11, 21], [11, 21, 16], [21, 16, 20], [16, 20, 14], [20, 14, 0], [14, 0, 25], [0, 25, 21], [25, 21, 11], [21, 11, 8], [11, 8, 30], [8, 30, 0], [30, 0, 15], [0, 15, 8], [15, 8, 27], [8, 27, 12], [27, 12, 0], [12, 0, 8], [0, 8, 0], [8, 0, 14], [0, 14, 21], [14, 21, 21], [21, 21, 11], [21, 11, 0], [11, 0, 11], [0, 11, 8], [11, 8, 30], [8, 30, 0], [30, 0, 25], [0, 25, 8], [25, 8, 17], [8, 17, 12], [17, 12, 0], [12, 0, 10], [0, 10, 8], [10, 8, 23], [8, 23, 12], [23, 12, 0], [12, 0, 24], [0, 24, 25], [24, 25, 8], [25, 8, 30], [8, 30, 0], [30, 0, 24], [0, 24, 8], [24, 8, 13], [8, 13, 12], [13, 12, 0], [12, 0, 14], [0, 14, 21], [14, 21, 21], [21, 21, 11], [21, 11, 0], [11, 0, 19], [0, 19, 21], [19, 21, 23], [21, 23, 20], [23, 20, 16], [20, 16, 20], [16, 20, 14], [20, 14

In [58]:
print(y)

[18, 21, 0, 15, 21, 28, 0, 8, 23, 12, 0, 30, 21, 26, 0, 11, 21, 16, 20, 14, 0, 25, 21, 11, 8, 30, 0, 15, 8, 27, 12, 0, 8, 0, 14, 21, 21, 11, 0, 11, 8, 30, 0, 25, 8, 17, 12, 0, 10, 8, 23, 12, 0, 24, 25, 8, 30, 0, 24, 8, 13, 12, 0, 14, 21, 21, 11, 0, 19, 21, 23, 20, 16, 20, 14, 0, 14, 21, 21, 11, 0, 20, 16, 14, 15, 25, 0, 25, 15, 8, 20, 17, 0, 30, 21, 26, 0, 13, 21, 23, 0, 30, 21, 26, 23, 0, 24, 26, 22, 22, 21, 23, 25, 0, 30, 21, 26, 0, 8, 23, 12, 0, 28, 12, 18, 10, 21, 19, 12, 0, 24, 12, 12, 0, 30, 21, 26, 0, 24, 21, 21, 20, 0, 15, 21, 22, 12, 0, 12, 27, 12, 23, 30, 25, 15, 16, 20, 14, 0, 28, 21, 23, 17, 24, 0, 21, 26, 25, 0, 13, 16, 20, 12, 0, 17, 12, 12, 22, 0, 14, 21, 16, 20, 14, 0, 21, 20, 12, 0, 24, 25, 12, 22, 0, 8, 25, 0, 8, 0, 25, 16, 19, 12, 0, 18, 12, 8, 23, 20, 16, 20, 14, 0, 20, 12, 28, 0, 25, 15, 16, 20, 14, 24, 0, 16, 24, 0, 13, 26, 20, 0, 25, 8, 17, 12, 0, 9, 23, 12, 8, 17, 24, 0, 11, 23, 16, 20, 17, 0, 28, 8, 25, 12, 23, 0, 24, 25, 8, 30, 0, 10, 21, 20, 24, 16, 24, 25, 1

In [59]:
X = np.array(X)
y = np.array(y)

In [60]:
# One hot encoding
X = tf.one_hot(X, vocab_size)

### Defining model + training + prediction

In [69]:
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, GRU, Dense

model = Sequential([
    Input(shape=(3, vocab_size)),
    GRU(32, return_sequences=True),
    GRU(32),
    Dense(vocab_size, activation="softmax")
])

model.compile(optimizer="adam", loss="sparse_categorical_crossentropy")

model.fit(X, y, epochs=80, verbose=0)

In [72]:
# Prediction function
def predict_next_char(text):
  inp = text
  inp_seq = np.array([[char2id[c] for c in inp]])
  inp_oh = tf.one_hot(inp_seq, vocab_size)
  inp = tf.convert_to_tensor(inp_oh)
  pred_id = np.argmax(model.predict(inp, verbose=0)[0])
  print("Next char prediction:", id2char[pred_id])

In [73]:
predict_next_char("mornin")

Next char prediction: g


In [78]:
predict_next_char("everythi")

Next char prediction: n


In [79]:
predict_next_char("hell")

Next char prediction: o


In [80]:
predict_next_char("welcom")

Next char prediction: e
